## Rent prediction model (RQ1)

Trains a gradient-boosted regression model to predict monthly rent from property
characteristics, benchmarked against a linear regression baseline.

Spatial location is represented as continuous coordinates rather than
categorical district labels, so that the model can generalise across districts
with few observations. Of 205 districts, the median contains four listings.

Inputs:
  data/processed/listings_clean.csv    2,400 London listings
  data/processed/district_spatial.csv  district centroids and distance to centre

Reported: MAE, RMSE, R2 and MAPE, against a linear regression baseline.

In [1]:
import pandas as pd
import numpy as np

listings = pd.read_csv('../data/processed/listings_clean.csv')
spatial = pd.read_csv('../data/processed/district_spatial.csv')

print("Listings:", listings.shape)
print("Spatial: ", spatial.shape)

Listings: (2400, 10)
Spatial:  (329, 6)


In [2]:
df = listings.merge(spatial,
                    left_on='subdistrict_code',
                    right_on='district',
                    how='left')

print("After join:", df.shape)
print("Listings with no spatial match:", df['easting'].isna().sum())

After join: (2400, 16)
Listings with no spatial match: 0


### Feature preparation

Location is represented by easting, northing and distance from central London
rather than by district label, following the approach agreed with the supervisor
to enable generalisation across sparse districts. Categorical property
attributes are one-hot encoded. The target is log-transformed rent; predictions
are converted back to pounds for reporting.

In [3]:
# Features the model will use
numeric = ['BEDROOMS', 'BATHROOMS', 'avg_distance_to_nearest_station',
           'easting', 'northing', 'dist_centre_km', 'is_shared']

categorical = ['PROPERTY TYPE', 'Furnish Type']

# One-hot encode the categorical columns
X = pd.get_dummies(df[numeric + categorical],
                   columns=categorical,
                   drop_first=True)

y = df['rent_log']

print("Features:", X.shape)
print(X.columns.tolist())

Features: (2400, 32)
['BEDROOMS', 'BATHROOMS', 'avg_distance_to_nearest_station', 'easting', 'northing', 'dist_centre_km', 'is_shared', 'PROPERTY TYPE_Bungalow', 'PROPERTY TYPE_Chalet', 'PROPERTY TYPE_Cottage', 'PROPERTY TYPE_Detached', 'PROPERTY TYPE_Duplex', 'PROPERTY TYPE_End of Terrace', 'PROPERTY TYPE_Flat', 'PROPERTY TYPE_Flat Share', 'PROPERTY TYPE_Ground Flat', 'PROPERTY TYPE_Ground Maisonette', 'PROPERTY TYPE_House', 'PROPERTY TYPE_House Share', 'PROPERTY TYPE_House of Multiple Occupation', 'PROPERTY TYPE_Maisonette', 'PROPERTY TYPE_Mews', 'PROPERTY TYPE_Mobile Home', 'PROPERTY TYPE_Penthouse', 'PROPERTY TYPE_Semi-Detached', 'PROPERTY TYPE_Studio', 'PROPERTY TYPE_Terraced', 'PROPERTY TYPE_Town House', 'Furnish Type_Furnished', 'Furnish Type_Furnished or unfurnished, landlord is flexible', 'Furnish Type_Part furnished', 'Furnish Type_Unfurnished']


### Train and test split

Twenty per cent of listings are held out for evaluation. The random seed is
fixed so that the split is reproducible across runs, allowing the ablation
comparisons in Section 4 to be made on identical data.

In [4]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Train:", X_train.shape)
print("Test: ", X_test.shape)

Train: (1920, 32)
Test:  (480, 32)


### Handling missing bathroom counts

Bathroom counts were set to missing for shared lettings, where the field
describes the whole property rather than the room being let. XGBoost handles
missing values natively by learning a default branch direction. Linear
regression cannot, so for the baseline only these values are imputed with the
training-set median. The imputation uses the training median rather than the
full dataset to avoid leaking test-set information.

In [5]:
# Linear regression cannot handle missing values; XGBoost can.
# For the baseline only, missing bathroom counts are imputed with the median.
X_train_lin = X_train.fillna(X_train['BATHROOMS'].median())
X_test_lin = X_test.fillna(X_train['BATHROOMS'].median())

print("Missing in baseline training set:", X_train_lin.isna().sum().sum())

Missing in baseline training set: 0


In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

baseline = LinearRegression()
baseline.fit(X_train_lin, y_train)
pred_log = baseline.predict(X_test_lin)

# Convert predictions and actuals from log scale back to pounds
pred_gbp = np.exp(pred_log)
actual_gbp = np.exp(y_test)

print("Linear regression baseline")
print("  R2:   ", round(r2_score(y_test, pred_log), 4))
print("  MAE:  £", round(mean_absolute_error(actual_gbp, pred_gbp)))
print("  RMSE: £", round(np.sqrt(mean_squared_error(actual_gbp, pred_gbp))))
print("  MAPE: ", round(np.mean(np.abs((actual_gbp - pred_gbp) / actual_gbp)) * 100, 1), "%")

Linear regression baseline
  R2:    0.7921
  MAE:  £ 560
  RMSE: £ 1284
  MAPE:  19.9 %


In [7]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    random_state=42,
)

model.fit(X_train, y_train)          # note: X_train, not X_train_lin
pred_log = model.predict(X_test)

pred_gbp = np.exp(pred_log)

print("XGBoost")
print("  R2:   ", round(r2_score(y_test, pred_log), 4))
print("  MAE:  £", round(mean_absolute_error(actual_gbp, pred_gbp)))
print("  RMSE: £", round(np.sqrt(mean_squared_error(actual_gbp, pred_gbp))))
print("  MAPE: ", round(np.mean(np.abs((actual_gbp - pred_gbp) / actual_gbp)) * 100, 1), "%")

XGBoost
  R2:    0.8225
  MAE:  £ 538
  RMSE: £ 1164
  MAPE:  18.3 %


In [8]:
results = pd.DataFrame([
    {'model': 'Linear regression', 'r2': 0.7921, 'mae': 560, 'rmse': 1284, 'mape': 19.9},
    {'model': 'XGBoost', 'r2': 0.8225, 'mae': 538, 'rmse': 1164, 'mape': 18.3},
])
results.to_csv('../results/rq1_baseline.csv', index=False)
print(results)

               model      r2  mae  rmse  mape
0  Linear regression  0.7921  560  1284  19.9
1            XGBoost  0.8225  538  1164  18.3


## Ablation 1: spatial representation

Compares continuous spatial features (easting, northing, distance from centre)
against 205 categorical district labels. Performance is reported separately for
high-density districts (E14, W8, SW5, SW7, W14) and long-tail districts with
fewer than five listings, to test whether continuous representation improves
generalisation where observations are sparse.

In [9]:
# Alternative representation: district as a categorical label
X_cat = pd.get_dummies(
    df[['BEDROOMS', 'BATHROOMS', 'avg_distance_to_nearest_station', 'is_shared',
        'PROPERTY TYPE', 'Furnish Type', 'subdistrict_code']],
    columns=['PROPERTY TYPE', 'Furnish Type', 'subdistrict_code'],
    drop_first=True,
)

print("Continuous representation:", X.shape[1], "features")
print("Categorical representation:", X_cat.shape[1], "features")

Continuous representation: 32 features
Categorical representation: 233 features


In [10]:
from sklearn.model_selection import train_test_split

# Identify district regimes
counts = df['subdistrict_code'].value_counts()
head_districts = ['E14', 'W8', 'SW5', 'SW7', 'W14']
tail_districts = counts[counts < 5].index.tolist()

print("Head districts:", len(head_districts), "|", counts[head_districts].sum(), "listings")
print("Tail districts:", len(tail_districts), "|", counts[tail_districts].sum(), "listings")

Head districts: 5 | 867 listings
Tail districts: 105 | 239 listings


In [11]:
def evaluate(X_features, label):
    """Train XGBoost on a feature set and report metrics overall and by regime."""
    Xtr, Xte, ytr, yte = train_test_split(
        X_features, y, test_size=0.2, random_state=42
    )

    m = XGBRegressor(n_estimators=500, learning_rate=0.05,
                     max_depth=6, random_state=42)
    m.fit(Xtr, ytr)
    pred = m.predict(Xte)

    # Which regime does each test listing belong to?
    test_districts = df.loc[Xte.index, 'subdistrict_code']
    in_head = test_districts.isin(head_districts)
    in_tail = test_districts.isin(tail_districts)

    def metrics(mask, name):
        if mask.sum() == 0:
            return
        a = np.exp(yte[mask])
        p = np.exp(pred[mask.values])
        print(f"  {name:12s} n={mask.sum():4d}  "
              f"R2={r2_score(yte[mask], pred[mask.values]):.4f}  "
              f"MAE=£{mean_absolute_error(a, p):.0f}  "
              f"MAPE={np.mean(np.abs((a-p)/a))*100:.1f}%")

    print(label)
    metrics(pd.Series(True, index=yte.index), "overall")
    metrics(in_head, "head")
    metrics(in_tail, "tail")
    print()


evaluate(X, "Continuous spatial features")
evaluate(X_cat, "Categorical district labels")

Continuous spatial features
  overall      n= 480  R2=0.8225  MAE=£538  MAPE=18.3%
  head         n= 175  R2=0.7657  MAE=£543  MAPE=16.7%
  tail         n=  57  R2=0.8345  MAE=£694  MAPE=21.9%

Categorical district labels
  overall      n= 480  R2=0.8172  MAE=£542  MAPE=18.4%
  head         n= 175  R2=0.7420  MAE=£581  MAPE=17.3%
  tail         n=  57  R2=0.8288  MAE=£721  MAPE=22.0%



In [12]:
def evaluate(X_features, label):
    """Train XGBoost on a feature set and return metrics by regime."""
    Xtr, Xte, ytr, yte = train_test_split(X_features, y, test_size=0.2, random_state=42)

    m = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42)
    m.fit(Xtr, ytr)
    pred = m.predict(Xte)

    test_districts = df.loc[Xte.index, 'subdistrict_code']
    regimes = {
        'overall': pd.Series(True, index=yte.index),
        'head': test_districts.isin(head_districts),
        'tail': test_districts.isin(tail_districts),
    }

    rows = []
    for name, mask in regimes.items():
        if mask.sum() == 0:
            continue
        a, p = np.exp(yte[mask]), np.exp(pred[mask.values])
        rows.append({
            'representation': label,
            'regime': name,
            'n': int(mask.sum()),
            'r2': round(r2_score(yte[mask], pred[mask.values]), 4),
            'mae': round(mean_absolute_error(a, p)),
            'mape': round(np.mean(np.abs((a - p) / a)) * 100, 1),
        })
    return pd.DataFrame(rows)


ablation1 = pd.concat([
    evaluate(X, 'Continuous'),
    evaluate(X_cat, 'Categorical'),
])
ablation1.to_csv('../results/rq1_ablation_spatial.csv', index=False)
print(ablation1)

  representation   regime    n      r2  mae  mape
0     Continuous  overall  480  0.8225  538  18.3
1     Continuous     head  175  0.7657  543  16.7
2     Continuous     tail   57  0.8345  694  21.9
0    Categorical  overall  480  0.8172  542  18.4
1    Categorical     head  175  0.7420  581  17.3
2    Categorical     tail   57  0.8288  721  22.0


In [13]:
from sklearn.model_selection import GroupKFold

def leave_district_out(X_features, label, n_splits=5):
    """
    Evaluate with entire districts held out, testing whether the model can
    price districts it has never seen. Categorical representations cannot
    generalise to unseen labels; continuous coordinates can interpolate.
    """
    groups = df['subdistrict_code']
    gkf = GroupKFold(n_splits=n_splits)

    r2s, maes, mapes = [], [], []

    for train_idx, test_idx in gkf.split(X_features, y, groups=groups):
        Xtr, Xte = X_features.iloc[train_idx], X_features.iloc[test_idx]
        ytr, yte = y.iloc[train_idx], y.iloc[test_idx]

        m = XGBRegressor(n_estimators=500, learning_rate=0.05,
                         max_depth=6, random_state=42)
        m.fit(Xtr, ytr)
        pred = m.predict(Xte)

        a, p = np.exp(yte), np.exp(pred)
        r2s.append(r2_score(yte, pred))
        maes.append(mean_absolute_error(a, p))
        mapes.append(np.mean(np.abs((a - p) / a)) * 100)

    return {
        'representation': label,
        'r2_mean': round(np.mean(r2s), 4),
        'r2_std': round(np.std(r2s), 4),
        'mae_mean': round(np.mean(maes)),
        'mape_mean': round(np.mean(mapes), 1),
    }


In [14]:


ablation2 = pd.DataFrame([
    leave_district_out(X, 'Continuous'),
    leave_district_out(X_cat, 'Categorical'),
])
ablation2.to_csv('../results/rq1_ablation_leave_district_out.csv', index=False)
print(ablation2)

  representation  r2_mean  r2_std  mae_mean  mape_mean
0     Continuous   0.7769  0.0241       645       20.2
1    Categorical   0.7042  0.0313       718       21.8


## Ablation 3: data curation decisions

Tests whether the curation decisions taken during cleaning measurably improved
model performance. Two can be tested directly from the cleaned dataset: masking
whole-property bathroom counts on shared lettings, and the binary indicator
distinguishing shared from whole-property markets. The Flat/Apartment
consolidation cannot be reversed from the cleaned output and is noted instead.

In [15]:
def quick_eval(X_features, label):
    """Standard split evaluation, returning overall metrics only."""
    Xtr, Xte, ytr, yte = train_test_split(X_features, y, test_size=0.2, random_state=42)
    m = XGBRegressor(n_estimators=500, learning_rate=0.05, max_depth=6, random_state=42)
    m.fit(Xtr, ytr)
    pred = m.predict(Xte)
    a, p = np.exp(yte), np.exp(pred)
    return {
        'variant': label,
        'r2': round(r2_score(yte, pred), 4),
        'mae': round(mean_absolute_error(a, p)),
        'mape': round(np.mean(np.abs((a - p) / a)) * 100, 1),
    }




In [16]:
# Variant A: without the is_shared indicator
X_no_shared = X.drop(columns=['is_shared'])

# Variant B: bathrooms restored for shares, using the median for whole properties
#            (approximates the uncorrected state; the original values are not
#             recoverable from the cleaned file)
X_bath_filled = X.copy()
X_bath_filled['BATHROOMS'] = X_bath_filled['BATHROOMS'].fillna(
    X_bath_filled['BATHROOMS'].median()
)



In [18]:
ablation3 = pd.DataFrame([
    quick_eval(X, 'Full cleaning (as applied)'),
    quick_eval(X_no_shared, 'Without is_shared indicator'),
    quick_eval(X_bath_filled, 'Bathrooms imputed rather than masked'),
])
ablation3.to_csv('../results/rq1_ablation_curation.csv', index=False)
print(ablation3)

                                variant      r2  mae  mape
0            Full cleaning (as applied)  0.8225  538  18.3
1           Without is_shared indicator  0.8213  534  18.3
2  Bathrooms imputed rather than masked  0.8146  566  18.8


## Ablation 4: leakage diagnostic

Deposit was excluded from the predictors because UK deposits are capped at five
weeks' rent under the Tenant Fees Act 2019, making the field a deterministic
transformation of the target. This diagnostic quantifies the artificial
improvement that would result from retaining it, confirming that its exclusion
is necessary for valid inference rather than merely cautious.

In [19]:
# Deposit was removed during cleaning; reload it from the raw file
raw = pd.read_csv('../data/raw/rent_ads_rightmove_extended.csv',
                  usecols=['address', 'rent', 'deposit'])

In [20]:
# Match on address and rent to recover the deposit for cleaned listings
# (address was dropped from the cleaned file, so rebuild from the cleaning steps)
print("Raw listings:", len(raw))
print("Deposits recorded as zero:", (raw['deposit'] == 0).sum())
print("Correlation, non-zero only:",
      round(raw[raw['deposit'] > 0][['rent', 'deposit']].corr().iloc[0, 1], 4))

Raw listings: 3478
Deposits recorded as zero: 1290
Correlation, non-zero only: 0.9633


In [21]:
# Load the variant retaining deposit
dep = pd.read_csv('../data/processed/listings_with_deposit.csv')
dep = dep.merge(spatial, left_on='subdistrict_code', right_on='district', how='left')

In [22]:
X_leak = pd.get_dummies(
    dep[numeric + categorical + ['deposit']],
    columns=categorical, drop_first=True
)

In [23]:
ablation4 = pd.DataFrame([
    quick_eval(X, 'Deposit excluded (as used)'),
    quick_eval(X_leak, 'Deposit retained (leakage)'),
])
print(ablation4)

                      variant      r2  mae  mape
0  Deposit excluded (as used)  0.8225  538  18.3
1  Deposit retained (leakage)  0.9266  305   9.7
